# 🧠 Module 4: Brain Tumor MRI Classification
**Dataset:** Brain Tumor MRI Dataset (Masoud Nickparvar) via Kaggle

**Classes:** Glioma | Meningioma | Pituitary | No Tumor (4 classes)

**Model:** EfficientNetB4 Transfer Learning + Grad-CAM

**Output:** `brain_mri_model.h5` → place in `ai-server/models/`

---
⚠️ **Recommended:** Run on Kaggle Notebooks or Google Colab (free T4 GPU)

In [ ]:
!pip install -q kaggle tensorflow matplotlib numpy opencv-python Pillow seaborn

In [ ]:
# ─── GPU CHECK ────────────────────────────────────────────────────────────────
import tensorflow as tf
print('TF Version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ─── STEP 2: Download Dataset ─────────────────────────────────────────────────
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p ./data --unzip
!ls ./data/

In [ ]:
# ─── STEP 3: Explore Dataset ──────────────────────────────────────────────────
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

BASE_DIR = './data'
# Dataset has Training/ and Testing/ folders
CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
CLASS_LABELS = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']

print('Dataset Structure:')
for split in ['Training', 'Testing']:
    total = 0
    for cls in CLASSES:
        path = os.path.join(BASE_DIR, split, cls)
        if os.path.exists(path):
            count = len(os.listdir(path))
            total += count
            print(f'  {split}/{cls}: {count}')
    print(f'  {split} TOTAL: {total}\n')

In [ ]:
# ─── STEP 4: Visualize Samples ────────────────────────────────────────────────
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(14, 14))

for i, cls in enumerate(CLASSES):
    imgs = [f for f in os.listdir(f'{BASE_DIR}/Training/{cls}') 
            if f.lower().endswith(('.jpg','.jpeg','.png'))][:4]
    for j, img_name in enumerate(imgs):
        img = mpimg.imread(f'{BASE_DIR}/Training/{cls}/{img_name}')
        axes[i][j].imshow(img, cmap='gray' if len(img.shape)==2 else None)
        if j == 0:
            axes[i][j].set_ylabel(CLASS_LABELS[i], fontsize=11, fontweight='bold')
        axes[i][j].axis('off')

plt.suptitle('Brain MRI Samples by Tumor Type', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('./data/mri_samples.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ─── STEP 5: Create Validation Split from Training ───────────────────────────
import shutil
from sklearn.model_selection import train_test_split

# Create train/val/test dirs
for split in ['train', 'val']:
    for cls in CLASSES:
        os.makedirs(f'./data/{split}/{cls}', exist_ok=True)

# Split Training → 85% train, 15% val
for cls in CLASSES:
    src = f'{BASE_DIR}/Training/{cls}'
    imgs = [f for f in os.listdir(src) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    train_imgs, val_imgs = train_test_split(imgs, test_size=0.15, random_state=42)
    
    for img in train_imgs:
        shutil.copy(f'{src}/{img}', f'./data/train/{cls}/{img}')
    for img in val_imgs:
        shutil.copy(f'{src}/{img}', f'./data/val/{cls}/{img}')
    print(f'{cls}: {len(train_imgs)} train, {len(val_imgs)} val')

print('\n✅ Split complete')

In [ ]:
# ─── STEP 6: Data Generators ──────────────────────────────────────────────────
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=False,
    brightness_range=[0.85, 1.15],
    shear_range=0.1
)
val_gen = ImageDataGenerator(rescale=1./255)

train_ds = train_gen.flow_from_directory(
    './data/train', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical'
)
val_ds = val_gen.flow_from_directory(
    './data/val', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical'
)
test_ds = val_gen.flow_from_directory(
    f'{BASE_DIR}/Testing', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

print('Class indices:', train_ds.class_indices)
NUM_CLASSES = len(train_ds.class_indices)

In [ ]:
# ─── STEP 7: Build EfficientNetB4 Model ───────────────────────────────────────
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras import layers, models, regularizers

def build_brain_model(num_classes=4):
    base = EfficientNetB4(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3)
    )
    base.trainable = False
    
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    return tf.keras.Model(inputs, outputs), base

model, base_model = build_brain_model(NUM_CLASSES)
model.summary()

In [ ]:
# ─── STEP 8: Phase 1 Training ─────────────────────────────────────────────────
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

os.makedirs('./output', exist_ok=True)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=True)]
)

callbacks = [
    EarlyStopping(patience=6, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-7),
    ModelCheckpoint('./output/brain_mri_phase1.h5', save_best_only=True, monitor='val_accuracy')
]

history1 = model.fit(train_ds, epochs=20, validation_data=val_ds, callbacks=callbacks)
print('Phase 1 complete. Val accuracy:', max(history1.history['val_accuracy']))

In [ ]:
# ─── STEP 9: Phase 2 Fine-Tuning ──────────────────────────────────────────────
base_model.trainable = True
# Freeze all but last 40 layers
for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-6),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks2 = [
    EarlyStopping(patience=8, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-9),
    ModelCheckpoint('./output/brain_mri_best.h5', save_best_only=True, monitor='val_accuracy')
]

history2 = model.fit(train_ds, epochs=30, validation_data=val_ds, callbacks=callbacks2)

In [ ]:
# ─── STEP 10: Evaluate ────────────────────────────────────────────────────────
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

y_pred = np.argmax(model.predict(test_ds), axis=1)
y_true = test_ds.classes

print('=== TEST RESULTS ===')
print(classification_report(y_true, y_pred, target_names=CLASS_LABELS))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', ax=axes[0],
            xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, cmap='Blues')
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

# Training curve
all_acc = history1.history['accuracy'] + history2.history['accuracy']
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
axes[1].plot(all_acc, label='Train Accuracy')
axes[1].plot(all_val_acc, label='Val Accuracy')
axes[1].set_title('Training Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('./data/mri_results.png', dpi=100)
plt.show()

In [ ]:
# ─── STEP 11: Save Final Model ────────────────────────────────────────────────
model.save('./output/brain_mri_model.h5')

# Save class mapping for ai-server
import json
class_map = {v: k for k, v in train_ds.class_indices.items()}
with open('./output/brain_mri_classes.json', 'w') as f:
    json.dump(class_map, f)

print('✅ Saved: ./output/brain_mri_model.h5')
print('✅ Saved: ./output/brain_mri_classes.json')
print('\n📁 COPY BOTH FILES TO: healthvision-ai/ai-server/models/')

In [ ]:
# ─── STEP 12: Test Prediction ─────────────────────────────────────────────────
from tensorflow.keras.models import load_model
import json

loaded_model = load_model('./output/brain_mri_model.h5')
with open('./output/brain_mri_classes.json') as f:
    class_map = json.load(f)

# Test on sample image
sample_class = CLASSES[0]
sample_imgs = [f for f in os.listdir(f'{BASE_DIR}/Testing/{sample_class}') 
               if f.lower().endswith(('.jpg','.jpeg','.png'))][:1]
img_path = f'{BASE_DIR}/Testing/{sample_class}/{sample_imgs[0]}'

img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
img_arr = tf.keras.utils.img_to_array(img) / 255.0
img_arr = np.expand_dims(img_arr, 0)

preds = loaded_model.predict(img_arr)[0]
pred_class = np.argmax(preds)

print('Probabilities:')
for i, (cls, prob) in enumerate(zip(CLASS_LABELS, preds)):
    marker = '←' if i == pred_class else ''
    print(f'  {cls}: {prob:.4f} {marker}')
print(f'\nPrediction: {CLASS_LABELS[pred_class]} | Confidence: {preds[pred_class]:.4f}')